### 01 — Generate the baseline

Makes 200 images with plain Stable Diffusion 1.5: 50 test prompts × 4 seeds.
This is the **control group** that the LoRA gets compared against later.

**Before running:** Runtime type is **T4 GPU**.
Images are saved to Google Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

#### 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/paint-by-pixel'      
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)

#### 2. Get the code from GitHub

In [ ]:
REPO = 'https://github.com/jesicag2/paint-by-pixel.git'
if os.path.exists('/content/paint-by-pixel'):
    !git -C /content/paint-by-pixel pull
else:
    !git clone {REPO} /content/paint-by-pixel
%cd /content/paint-by-pixel
!git log --oneline | head -5

#### 3. Install packages
`torchao` is removed because it conflicts with newer libraries.

In [ ]:
!pip uninstall -y -q torchao
%pip install -q -r requirements.txt
%pip install -q -e .        # makes `import paintbypixel` work

#### 4. Point `outputs/` at Drive
The scripts save to `outputs/eval/...`. Folder lives on Drive.

In [ ]:
if not os.path.islink('outputs'):
    import shutil; shutil.rmtree('outputs', ignore_errors=True)
    os.symlink(f'{DRIVE}/outputs', 'outputs')
!ls -la outputs

#### 5. Quick test: 3 prompts (12 images)

In [ ]:
!python eval/generate_eval_set.py --variant base --limit 3

#### 6. Full baseline: all 50 prompts

In [ ]:
!python eval/generate_eval_set.py --variant base

#### 7. Look at the results

In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

out = 'outputs/eval/raw/base'
man = pd.read_csv(f'{out}/manifest.csv')
print(len(man), 'images | avg', round(man.seconds.mean(), 2), 's each | safety-filter blanks:', man.nsfw_flag.sum())

seed0 = man[man.seed == 0].sort_values('prompt_id')
fig, axes = plt.subplots(5, 10, figsize=(20, 11))
for ax, (_, r) in zip(axes.flat, seed0.iterrows()):
    ax.imshow(Image.open(f"{out}/{r.file}")); ax.set_title(r.subject[:18], fontsize=7); ax.axis('off')
plt.tight_layout(); plt.show()